# Calculating Power-law scaling in fluctuations of information

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = ''

In [3]:
DATA_PATH = '../../../updated-data/obs/lme-ready'
NULL_DATA_PATH = '../../../updated-data/null/null-lme-ready'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [5]:
def lang_id(x):
    if 'xling-' in x:
        return x.split('-')[2]
    else:
        return 'eng'

def corpus_id(x):
    if 'xling-' in x:
        return x.split('-')[1]
    else:
        return x.split('-')[0]

In [6]:
fs = [
    f for f in os.listdir(DATA_PATH)
    if (('xling-' in f) or ('callfriend' in f) or ('callhome' in f))
    and (not f.startswith('.'))
    and f.endswith('.parquet')
]
len(fs)

1289

## Processing individual datasets

In [7]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [8]:
COEF_VAR = 'AVAR'

In [9]:
param_list = 'time_delta'

In [25]:
df_scale, df_regression, k_docs, error_tracker = [], [], dict(), 0

In [26]:
for f in tqdm(fs):
    # print('===][===')
    # print(f.split('/')[-1])

    table = pq.ParquetFile(os.path.join(DATA_PATH, f))

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id',
        # 'nx', 'ny'
    ]).to_pandas()
    sel = df['y_turn_id'] - df['x_turn_id']

    df['self'] = (df['x_speaker'] == df['y_speaker'])
    # df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]


    # find and eliminate bad y_turns
    #   The idea is that some speakers only generate one utterance per
    #   conversation, and thus grouping the way we're doing here yields
    #   errors when calc'ing power-law exponents
    #
    #   (1) .drop dupes on y_speaker, y_turn_id
    #   (2) ["y_speaker"].count_values()
    #   (3) k > 1

    df = df.loc[
        (~df[COEF_VAR].isna())
        & (sel > 0) & (sel <= 200)
    ]

    groups = df.groupby(by=['x_speaker', 'y_speaker'])

    for labels, dfi in groups:
        self_other = dfi['self'].unique()[0]
        dfi['min_y_turn_id'] = dfi.groupby('x_turn_id')['y_turn_id'].transform('min')

        try:
            if len(dfi) > 1:

                b, a = np.polyfit(
                    np.log((dfi['y_turn_id'] - dfi['min_y_turn_id']).values + 1),
                    np.log(dfi['AVAR'].values + 1e-9),
                    1
                )

                df_regression += [
                    {
                        'corpus': corpus_id(f),
                        'lang': lang_id(f),
                        'cid': f.replace('.parquet', ''),
                        'speaker': labels[0],
                        'self': self_other,
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

        except Exception:
            error_tracker += 1

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/1289 [00:00<?, ?it/s]

** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  5 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  5 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number 

In [27]:
len(df_regression), error_tracker

(7929, 31)

In [ ]:
# error_tracker[0].head()

In [ ]:
# error_tracker[1].head()

In [ ]:
# df_regression.isna().sum()

In [ ]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [32]:
df_regression.to_parquet(
    os.path.join(OUTPUTS_PATH,'all.parquet'),
    compression='snappy', engine='fastparquet'
)

## Corpus level analysis of results

In [28]:
split_by = ['lang', 'self']

In [29]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [30]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t',  'k']].values]

In [31]:
df0.head(n=20)

,lang,self,b,std,k,se,t,p
0,cro,False,-0.187153,0.361883,1796,0.008539,-21.916980,7.621735e-95
1,cro,True,-0.279354,0.335203,567,0.014077,-19.844427,3.209277e-67
2,deu,False,-0.314837,0.214959,312,0.012170,-25.870710,8.030880e-80
3,deu,True,-0.355027,0.203456,258,0.012667,-28.028563,2.016935e-80
4,eng,False,-0.278852,0.267248,534,0.011565,-24.111760,1.039029e-87
5,eng,True,-0.363434,0.228547,448,0.010798,-33.658078,6.301494e-125
6,fin,False,-0.557864,0.508209,166,0.039445,-14.142951,1.482552e-30
7,fin,True,-0.535022,0.742224,166,0.057608,-9.287318,4.486423e-17
8,fra,False,-0.185181,0.382884,175,0.028943,-6.398042,7.035003e-10
9,fra,True,-0.233752,0.494461,127,0.043876,-5.327522,2.214882e-07


In [33]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'language-results.csv'),
    index=False, encoding='utf-8'
)

## Aggregate analysis

In [34]:
split_by = ['self']

In [35]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [36]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t',  'k']].values]

In [37]:
df0.head(n=20)

,self,b,std,k,se,t,p
0,False,-0.231627,0.373382,5003,0.005279,-43.878460,0.0
1,True,-0.333884,0.324017,2926,0.005990,-55.739834,0.0


In [38]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'agg-results.csv'),
    index=False,
    encoding='utf-8'
)